# LTX-2.3 22B Distilled · Kaggle T4 x2 一键册

和上一册同构：**配置 → 环境（固定 Wan2GP 到 5 月 commit + 7 处 dtype 补丁）→ 权重 → 启动**。

| 项 | 说明 |
|---|---|
| 加速器 | **GPU T4 x2** |
| Internet | 打开 |
| 权重 | 约 26 GB → `/kaggle/tmp/models`，再软链进 `Wan2GP/models` |
| 引擎 | `run_ltx23.py`（GGUF Q4_K_M 参考路径，transformer 整块放 GPU0、Gemma 常驻 GPU1） |
| 精度 | transformer FP16（T4 张量核比 BF16 快约 3×），视频 VAE 强制 FP32 |

**为什么必须固定 Wan2GP**：更晚的版本改了 LTX-2 的代码路径，配这个 GGUF 会在 FP16 溢出（黑视频）。
本册 commit = `b8880bd5d292`（2026-05-13）。

**Secret**：同样优先 Kaggle Secrets，其次 Step 0 硬编码；只需要 `HF_TOKEN`（可选）。


In [ ]:
# ============ 0. 配置（只改这一段） ============
import os, sys, json, math, time, glob, shutil, subprocess

IS_KAGGLE = os.path.isdir("/kaggle")
WORK = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./kaggle_working")
TMP  = "/kaggle/tmp"     if IS_KAGGLE else os.path.abspath("./kaggle_tmp")
os.makedirs(WORK, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

# ---- Secrets：先读 Kaggle Secrets，取不到就用下面的硬编码值 ----
HARDCODED_SECRETS = {
    "HF_TOKEN": "",            # 私有/限流仓库才需要
    "KAGGLE_USERNAME": "",
    "KAGGLE_KEY": "",
    "CF_PRIVATE_HOST": "",
    "CF_TUNNEL_TOKEN": "",
}

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            print(f"🔑 {name}: 来自 Kaggle Secrets")
            return v
    except Exception:
        pass
    v = HARDCODED_SECRETS.get(name, "")
    print(f"🔑 {name}: {'使用硬编码值' if v else '未设置（可选）'}")
    return v

HF_TOKEN = secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "CF_PRIVATE_HOST", "CF_TUNNEL_TOKEN"):
    _v = secret(_k)
    if _v:
        os.environ[_k] = _v

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- 仓库 / Wan2GP ----
REPO_URL   = "https://github.com/milleniummike-del/kaggle-minimax.git"
REPO_DIR   = os.path.join(WORK, "kaggle-minimax")
WAN2GP_DIR = os.path.join(WORK, "Wan2GP")
WAN2GP_URL = "https://github.com/DeepBeepMeep/Wan2GP.git"
WAN2GP_COMMIT = "b8880bd5d292edfbe2384f5c951d95757ad59ea0"

print("=" * 70)
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"], text=True).strip()
    for _i, _l in enumerate(smi.splitlines()):
        print(f"✅ GPU {_i}: {_l}")
    if len(smi.splitlines()) < 2:
        print("⚠️ 本册需要 2 块 GPU：Settings → Accelerator → GPU T4 x2")
except Exception:
    print("⚠️ 没检测到 GPU（下载/环境步骤仍可跑）")
print(f"💾 {TMP} 可用 {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
print("=" * 70)


In [ ]:
# ============ 1. 环境：拉仓库 + 固定 Wan2GP + 装依赖 ============
import os, sys, subprocess

def _run(cmd, **kw):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

# 1a. 引擎仓库（每册都从这里取 frontend.py / run_ltx*.py）
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _run(["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", "HEAD"])
    _run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    _run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR])
print("✅ 仓库就绪:", REPO_DIR)

# 1b. Wan2GP 固定到引擎适配的 commit（换版本会改变算子/精度路径）
if not os.path.isdir(os.path.join(WAN2GP_DIR, ".git")):
    shutil.rmtree(WAN2GP_DIR, ignore_errors=True)
    os.makedirs(WAN2GP_DIR, exist_ok=True)
    _run(["git", "-C", WAN2GP_DIR, "init", "-q"])
    _run(["git", "-C", WAN2GP_DIR, "remote", "add", "origin", WAN2GP_URL])
_run(["git", "-C", WAN2GP_DIR, "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT])
_run(["git", "-C", WAN2GP_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]}")

# 1d. Wan2GP 运行时补丁（这些补丁就是仓库 ltx/cell1.py 里的 7 处，直接复用它的源码）
_src = open(os.path.join(REPO_DIR, "ltx", "cell1.py"), encoding="utf-8").read()
_src = _src.replace("print(\"Running cell1.py\")", "")
os.chdir(WORK)                      # cell1.py 里用的是相对路径 'Wan2GP'
exec(compile(_src, "ltx_cell1.py", "exec"), {"__name__": "__main__"})
print("✅ LTX 环境就绪")


In [ ]:
# ============ 2. 权重 → /kaggle/tmp/models，再软链回 Wan2GP/models ============
# 仓库自带 downloadltx.py 的文件清单；这里额外优先复用 /kaggle/input 里挂载的同名文件。
from huggingface_hub import hf_hub_download

LTX_REPO   = "DeepBeepMeep/LTX-2"
MODELS_TMP = os.path.join(TMP, "models")
LINK_DIR   = os.path.join(WAN2GP_DIR, "models")
os.makedirs(MODELS_TMP, exist_ok=True)
os.makedirs(LINK_DIR, exist_ok=True)

GEMMA = "gemma-3-12b-it-qat-q4_0-unquantized"
MAIN_FILES = [
    "ltx-2.3-22b-distilled-Q4_K_M_light.gguf",
    "ltx-2.3-22b_vae.safetensors",
    "ltx-2.3-22b_audio_vae.safetensors",
    "ltx-2.3-22b_vocoder.safetensors",
    "ltx-2.3-22b_text_embedding_projection.safetensors",
    "ltx-2.3-22b_embeddings_connector.safetensors",
    "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    "scenema-audio-transformer_quanto_bf16_int8.safetensors",
]
GEMMA_FILES = [
    "gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors",
    "added_tokens.json", "chat_template.json", "config_light.json",
    "generation_config.json", "preprocessor_config.json", "processor_config.json",
    "special_tokens_map.json", "tokenizer.json", "tokenizer.model", "tokenizer_config.json",
]

_INDEX = {}
if os.path.isdir("/kaggle/input"):
    for _root, _, _files in os.walk("/kaggle/input"):
        for _f in _files:
            _INDEX.setdefault(_f, os.path.join(_root, _f))

def ensure(hf_path):
    dest = os.path.join(MODELS_TMP, hf_path)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    src = _INDEX.get(os.path.basename(hf_path))
    if src:
        os.symlink(src, dest)
        print(f"  ⚡ 复用 Dataset {hf_path}")
    else:
        print(f"  ⬇️ {hf_path}")
        hf_hub_download(repo_id=LTX_REPO, filename=hf_path, local_dir=MODELS_TMP,
                        token=os.environ.get("HF_TOKEN") or None)
    return dest

for _f in MAIN_FILES:
    ensure(_f)
for _f in GEMMA_FILES:
    ensure(f"{GEMMA}/{_f}")

# Wan2GP 只从自己的 models/ 找文件：软链过去（实体仍在 /kaggle/tmp，不占 working 配额）
for _f in MAIN_FILES:
    _link = os.path.join(LINK_DIR, _f)
    if not os.path.exists(_link):
        os.symlink(os.path.join(MODELS_TMP, _f), _link)
_gemma_link = os.path.join(LINK_DIR, GEMMA)
if not os.path.exists(_gemma_link):
    os.symlink(os.path.join(MODELS_TMP, GEMMA), _gemma_link)

free = shutil.disk_usage(TMP).free / 1e9
print(f"\n✅ LTX-2.3 权重就绪（全套约 26 GB）| /kaggle/tmp 剩余 {free:.0f} GB")


In [ ]:
# ============ 3. 启动引擎（跑完会出现 Cloudflare / Gradio 公网链接） ============
# 这一格会一直占着，日志会实时刷出来；中断内核或点停止即可关服务，重跑本格=重启。
import codecs, re, time
import psutil

ENGINE_DIR = WORK
ENGINE     = os.path.join(REPO_DIR, "run_ltx23.py")
EXTRA_ENV  = {}

assert os.path.isfile(ENGINE), f"引擎不存在：{ENGINE}（先跑 Step 1）"
for _k, _v in list(EXTRA_ENV.items()):
    if _k.endswith("_DIR"):
        assert os.path.exists(_v), f"缺目录：{_v}（先跑 Step 2）"

# 清掉上一轮残留的引擎 / cloudflared，避免重复占显存
for _p in psutil.process_iter(["name", "cmdline"]):
    try:
        _args = _p.info["cmdline"] or []
        _name = (_p.info["name"] or "").lower()
        _hit = any(os.path.basename(a) == os.path.basename(ENGINE) for a in _args[1:3])
        if _p.pid != os.getpid() and (_hit or _name == "cloudflared"):
            _p.kill()
    except Exception:
        pass
time.sleep(1)

ENV = os.environ.copy()
ENV.update({k: str(v) for k, v in EXTRA_ENV.items()})
print(f"🚀 cd {ENGINE_DIR} && python {os.path.basename(ENGINE)}")
sys.stdout.flush()

proc = subprocess.Popen([sys.executable, ENGINE], cwd=ENGINE_DIR, env=ENV,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()
        out = "".join(parts)
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        print(f"\n❌ 引擎退出（code={rc}）。code=-9 通常是主机内存不够。")
except KeyboardInterrupt:
    proc.kill()
    proc.wait()
    print("🛑 已停止。重跑本格即可再启动。")
